# KaBuM — Scraper Multi-Categoria
Coleta dados de CPU, GPU, Placa-mãe, SSD, Fonte e RAM usando `__NEXT_DATA__`.

In [2]:
import requests
import json
import time
import random
import pathlib
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

In [4]:
CATEGORIAS = {
    "cpu":       "https://www.kabum.com.br/hardware/processadores",
    "gpu":       "https://www.kabum.com.br/hardware/placa-de-video-vga",
    "placa_mae": "https://www.kabum.com.br/hardware/placas-mae",
    "ssd":       "https://www.kabum.com.br/hardware/ssd-2-5",
    "fonte":     "https://www.kabum.com.br/hardware/fontes",
    "ram":       "https://www.kabum.com.br/hardware/memoria-ram",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
}

DATA_COLETA = date.today().isoformat()
NUM_PAGINAS = 20

In [5]:
def extrair_produtos(html):
    """Extrai a lista de produtos do __NEXT_DATA__ da página."""
    soup = BeautifulSoup(html, "html.parser")
    script_next = soup.find("script", id="__NEXT_DATA__")
    catalog = json.loads(json.loads(script_next.string)["props"]["pageProps"]["data"])
    return catalog["catalogServer"]["data"]


def parsear_produto(produto, categoria_key):
    """Mapeia os campos do produto para o formato do DataFrame."""
    return {
        "data_coleta":     DATA_COLETA,
        "categoria_key":   categoria_key,
        "id":              produto.get("code"),
        "nome":            produto.get("name"),
        "preco_original":  produto.get("oldPrice"),
        "preco_atual":     produto.get("price"),
        "preco_pix":       produto.get("priceWithDiscount"),
        "desconto_pct":    produto.get("discountPercentage"),
        "avaliacao":       produto.get("rating"),
        "num_avaliacoes":  produto.get("ratingCount"),
        "disponivel":      produto.get("available"),
        "fabricante":      produto.get("manufacturer", {}).get("name"),
        "categoria":       produto.get("category"),
        "garantia":        produto.get("warranty"),
        "frete_gratis":    produto.get("flags", {}).get("isFreeShipping"),
        "url":             f"https://www.kabum.com.br/produto/{produto.get('code')}/{produto.get('friendlyName', '')}",
    }


def scrape_categoria(nome, url, num_paginas=NUM_PAGINAS):
    """Raspa todas as páginas de uma categoria."""
    todos = []
    print(f"\n{'='*50}")
    print(f"Coletando: {nome.upper()} — {url}")

    for pagina in range(1, num_paginas + 1):
        url_pag = f"{url}?page_number={pagina}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            produtos_raw = extrair_produtos(resp.text)
            if not produtos_raw:
                print(f"  Página {pagina}: sem produtos — encerrando")
                break
            todos.extend([parsear_produto(p, nome) for p in produtos_raw])
            print(f"  Página {pagina}: {len(produtos_raw)} produtos (total: {len(todos)})")
        except Exception as e:
            print(f"  Página {pagina}: ERRO — {e}")
            break
        time.sleep(random.uniform(0.5, 1.5))

    print(f"  Total final: {len(todos)} produtos")
    return todos

In [6]:
data_dir = pathlib.Path(r"C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva-dado\Projetos\Projeto 1\00_Dados") / DATA_COLETA
data_dir.mkdir(exist_ok=True)

resultados = {}

for nome, url in CATEGORIAS.items():
    produtos = scrape_categoria(nome, url)
    df = pd.DataFrame(produtos)
    resultados[nome] = df
    arquivo = data_dir / f"kabum_{nome}_{DATA_COLETA}.csv"
    df.to_csv(arquivo, index=False, encoding="utf-8-sig")
    print(f"  Salvo: {arquivo}")

print("\n✓ Coleta concluída!")


Coletando: CPU — https://www.kabum.com.br/hardware/processadores
  Página 1: 60 produtos (total: 60)
  Página 2: 60 produtos (total: 120)
  Página 3: 60 produtos (total: 180)
  Página 4: 60 produtos (total: 240)
  Página 5: 60 produtos (total: 300)
  Página 6: 60 produtos (total: 360)
  Página 7: 60 produtos (total: 420)
  Página 8: 60 produtos (total: 480)
  Página 9: 2 produtos (total: 482)
  Página 10: sem produtos — encerrando
  Total final: 482 produtos
  Salvo: C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva-dado\Projetos\Projeto 1\00_Dados\2026-05-16\kabum_cpu_2026-05-16.csv

Coletando: GPU — https://www.kabum.com.br/hardware/placa-de-video-vga
  Página 1: 60 produtos (total: 60)
  Página 2: 60 produtos (total: 120)
  Página 3: 60 produtos (total: 180)
  Página 4: 60 produtos (total: 240)
  Página 5: 60 produtos (total: 300)
  Página 6: 60 produtos (total: 360)
  Página 7: 60 produtos (total: 420)
  Página 8: 60 produtos (total: 480)
  Página

In [7]:
# Consolida tudo num único CSV
df_consolidado = pd.concat(resultados.values(), ignore_index=True)
arquivo_consolidado = data_dir / f"kabum_todas_pecas_{DATA_COLETA}.csv"
df_consolidado.to_csv(arquivo_consolidado, index=False, encoding="utf-8-sig")

print(f"CSV consolidado: {arquivo_consolidado}")
print(f"Total de produtos: {len(df_consolidado)}\n")
print(df_consolidado.groupby("categoria_key").size().rename("qtd_produtos"))

CSV consolidado: C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva-dado\Projetos\Projeto 1\00_Dados\2026-05-16\kabum_todas_pecas_2026-05-16.csv
Total de produtos: 4636

categoria_key
cpu           482
fonte         616
gpu           629
placa_mae     842
ram          1200
ssd           867
Name: qtd_produtos, dtype: int64


In [ ]:
# Verificação rápida — 3 primeiros de cada categoria
colunas = ["nome", "preco_pix", "fabricante", "disponivel"]
for nome, df in resultados.items():
    print(f"\n--- {nome.upper()} ---")
    display(df[colunas].head(3))